# 03a · Unsupervised models: first run on v0 and the tuning log

Owner: Arva (A9–A10). Isolation Forest and the autoencoder from `src/models_unsupervised.py` on `data/processed/features_v0.parquet`, evaluated on the **val** split only (TEAM_PLAN 5.6: never tune on test). Every run adds one row to the tuning log `results/tables/tuning_log_unsupervised.csv`: parameters, val ROC-AUC, val PR-AUC, val recall when the top 1% of val user-days are flagged, and training time. The best run is the one with the highest **val PR-AUC**.

On Mon Nov 9 (A15) the same cells run on `features_v1.parquet`: change `DATASET` below.

Run with Restart & Run All. The log is rebuilt from scratch on every full run.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd

from src import models_unsupervised as mu
from src.config import PROCESSED_DIR, TABLES_DIR

DATASET = 'v0'
df = pd.read_parquet(PROCESSED_DIR / f'features_{DATASET}.parquet')
log_path = TABLES_DIR / mu.TUNING_LOG
log_path.unlink(missing_ok=True)       # rebuilt from scratch on every Restart & Run All
X, y, cols = mu.get_model_matrix(df, split='train')
print(f'{DATASET}: {len(df):,} user-days, {len(cols)} features, '
      f'{int(((df.split == "train") & (df.label == 0)).sum()):,} normal training rows')
df.groupby('split').label.agg(['size', 'sum']).rename(columns={'size': 'rows', 'sum': 'malicious'})

v0: 31,175 user-days, 7 features, 20,320 normal training rows


,rows,malicious
split,,
test,5688,40
train,20459,139
val,5028,57


## A9: first run with the card settings

Isolation Forest (A5 settings) and the autoencoder (A6 architecture), each scored on every row and saved as `results/scores/<model>_v0.parquet` (the final `<model>.parquet` files come from v1 in A16).

In [2]:
first = []
for name in ('iforest', 'autoencoder'):
    log, _ = mu.tune(df, name, grid={}, dataset=DATASET)          # an empty grid = the defaults
    scores, info = mu.train_and_score(df, name, save_as=f'{name}_{DATASET}')
    first.append(log.assign(saved=str(info['path'].relative_to(ROOT))))
    if name == 'autoencoder':
        fig = mu.plot_loss_curves(info['detector'].history, f'autoencoder_loss_{DATASET}')
pd.concat(first)[['model', 'params', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'fit_seconds', 'saved']].round(4)

,model,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds,saved
0,iforest,{},0.8736,0.0542,0.0526,0.29,results\scores\iforest_v0.parquet
0,autoencoder,{},0.8632,0.1037,0.1228,8.58,results\scores\autoencoder_v0.parquet


## A10: grid search

- **Isolation Forest:** `n_estimators` {100, 200, 500} × `max_samples` {256, 1024, auto} × `max_features` {0.5, 1.0} = 18 runs.
- **Autoencoder:** bottleneck {8, 16} × hidden layers {(64, 32), (128, 64)} × dropout {0, 0.1} × learning rate {1e-3, 1e-4} = 16 runs.

In [3]:
if_log, if_best = mu.tune(df, 'iforest', dataset=DATASET)
print('best Isolation Forest (val PR-AUC):', if_best)
if_log.sort_values('val_pr_auc', ascending=False).head(5).round(4)

best Isolation Forest (val PR-AUC): {'max_features': 0.5, 'max_samples': 1024, 'n_estimators': 500}


,model,dataset,train_on,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds,n_train,n_features
5,iforest,v0,normal,"{""max_features"": 0.5, ""max_samples"": 1024, ""n_...",0.8764,0.0625,0.0877,0.59,20320,7
4,iforest,v0,normal,"{""max_features"": 0.5, ""max_samples"": 1024, ""n_...",0.8750,0.0610,0.1053,0.26,20320,7
7,iforest,v0,normal,"{""max_features"": 0.5, ""max_samples"": ""auto"", ""...",0.8700,0.0605,0.0877,0.25,20320,7
1,iforest,v0,normal,"{""max_features"": 0.5, ""max_samples"": 256, ""n_e...",0.8700,0.0605,0.0877,0.26,20320,7
3,iforest,v0,normal,"{""max_features"": 0.5, ""max_samples"": 1024, ""n_...",0.8744,0.0603,0.0877,0.14,20320,7


In [4]:
ae_log, ae_best = mu.tune(df, 'autoencoder', dataset=DATASET)
print('best autoencoder (val PR-AUC):', ae_best)
ae_log.sort_values('val_pr_auc', ascending=False).head(5).round(4)

best autoencoder (val PR-AUC): {'bottleneck': 16, 'dropout': 0.0, 'hidden': (128, 64), 'lr': 0.0001}


,model,dataset,train_on,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds,n_train,n_features
11,autoencoder,v0,normal,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.9109,0.1221,0.1579,8.70,20320,7
10,autoencoder,v0,normal,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.8782,0.1185,0.1754,6.43,20320,7
0,autoencoder,v0,normal,"{""bottleneck"": 8, ""dropout"": 0.0, ""hidden"": [6...",0.8929,0.1116,0.1404,6.12,20320,7
8,autoencoder,v0,normal,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.8632,0.1037,0.1228,6.91,20320,7
2,autoencoder,v0,normal,"{""bottleneck"": 8, ""dropout"": 0.0, ""hidden"": [1...",0.8860,0.1004,0.1754,3.70,20320,7


## The tuning log so far

In [5]:
log = pd.read_csv(log_path)
print(f'{len(log)} runs in {log_path.relative_to(ROOT)}')
best = log.loc[log.groupby('model').val_pr_auc.idxmax()]
best[['model', 'dataset', 'params', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'fit_seconds']].round(4)

36 runs in results\tables\tuning_log_unsupervised.csv


,model,dataset,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds
31,autoencoder,v0,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.9109,0.1221,0.1579,8.70
7,iforest,v0,"{""max_features"": 0.5, ""max_samples"": 1024, ""n_...",0.8764,0.0625,0.0877,0.59


**Notes (Arva):** what the numbers say about v0 and which settings to carry to v1. *Human-only: write your own reading here.*